In [3]:
from langchain_core.documents import Document


In [4]:
#exmple
doc=Document(
  page_content="ich bin ali",
  metadata={
    "source":"exmple.pdf",
    "pages":1,
    "autor":"behrooz filzade"
    "data_create""2026-01-01"
    
  }
)

In [5]:
import os
os.makedirs("../data/text_files",exist_ok=True)

In [6]:
from langchain_community.document_loaders import TextLoader 
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
 
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python emphasizes clean syntax, which allows developers to write code that is easy to understand and maintain.\n\nIt supports multiple programming paradigms, including procedural, object-oriented, and functional programming. This flexibility makes it suitable for a wide range of applications, such as web development, data analysis, artificial intelligence, machine learning, automation, and scientific computing.\n\nOne of Python’s greatest strengths is its extensive ecosystem of libraries and frameworks. Tools like NumPy, Pandas, TensorFlow, Django, and Flask enable developers to build complex systems efficiently without reinventing the wheel.\n\nPython also has a large and active community, which contributes to its continuous improvement and pr

In [7]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader,DirectoryLoader


dir_loader=DirectoryLoader(
  "../data/pdf",
   glob="**/*.pdf",
   loader_cls=PyMuPDFLoader,
  
   show_progress=True 
)

pdf_document=dir_loader.load()



100%|██████████| 3/3 [00:00<00:00, 12.75it/s]


### embedding and vectorstoredb

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()  # Fixed: Added underscore to match the method name
        
    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)  # Fixed: PascalCase class name
            print(f"Model loaded successfully.")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: list[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args: 
            texts: List of text strings to embed.

        Returns:
            Numpy array of embeddings with shape (len(texts), embedding_dim).
        """
        if not self.model:
            raise ValueError("Model not loaded")  # Fixed: Corrected ValueError spelling
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings  # Fixed: Corrected spelling

### initialize the embedding manager

embedding_manager= EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


100%|██████████| 3/3 [00:20<00:00,  3.42it/s]

Model loaded successfully.


###vectorstoredb

In [10]:
import os
import uuid
import numpy as np
import chromadb
from typing import List, Any, Dict
from chromadb.config import Settings

class VectorStore:
    """Handles vector database operations using ChromaDB"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.
        
        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Path to persist the vector store data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.
        
        Args:
            documents: List of document objects (e.g., LangChain Document objects).
            embeddings: Corresponding numpy embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata) if hasattr(doc, 'metadata') else {}
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content) if hasattr(doc, 'page_content') else len(str(doc))
            metadatas.append(metadata)
            
            # Document content
            content = doc.page_content if hasattr(doc, 'page_content') else str(doc)
            documents_text.append(content)
            
            # Embedding (convert numpy array to list for ChromaDB)
            embeddings_list.append(embedding.tolist())
            
        # Add to collection 
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

# Initialize the vector store manager
vector_store_manager = VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 444


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size: int = 1000, chunk_overlap: int = 100):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Args:
        documents: List of LangChain document objects.
        chunk_size: Maximum size of each chunk (in characters).
        chunk_overlap: Number of characters to overlap between chunks.
        
    Returns:
        List of split document chunks.
    """
    # Fixed: Corrected class name and parameter spellings
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

# Usage:
chunks = split_documents(pdf_document)

Split 37 documents into 222 chunks

Example chunk:
Content: © 2019 IJRAR June 2019, Volume 6, Issue 2                                           www.ijrar.org  (E-ISSN 2348-1269, P- ISSN 2349-5138) 
IJRAR1ARP035 
International Journal of Research and Analytical...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2019-09-27T13:46:56+05:30', 'source': '..\\data\\pdf\\IJRAR1ARP035.pdf', 'file_path': '..\\data\\pdf\\IJRAR1ARP035.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': 'Students', 'subject': '', 'keywords': '', 'moddate': '2019-09-27T13:46:56+05:30', 'trapped': '', 'modDate': "D:20190927134656+05'30'", 'creationDate': "D:20190927134656+05'30'", 'page': 0}


In [12]:
###convert the text to embeddings
texts=[doc.page_content for doc in chunks]
###Generate the Embeddings
embedding_manager=EmbeddingManager()
embeddings= embedding_manager.generate_embeddings(texts)

##store in to the vectordatabase 
vector_store_manager.add_documents(chunks,embeddings)


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2223.00it/s]


Model loaded successfully.
Generating embeddings for 222 texts...


Batches: 100%|██████████| 7/7 [00:28<00:00,  4.02s/it]


Generated embeddings with shape: (222, 384)
Adding 222 documents to vector store...
Successfully added 222 documents to vector store
Total documents in collection: 666


In [13]:
from typing import List, Dict, Any

class RAGRetriever:
    """
    Handles query-based retrieval from the vector store.
    """
    def __init__(self, vector_store, embedding_manager):
        """
        Initialize the retriever.
        
        Args:
            vector_store: Vector store manager instance containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.2) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.
        
        Args:
            query: The search query string.
            top_k: Number of nearest neighbors to retrieve.
            score_threshold: Minimum similarity score threshold.
            
        Returns:
            List of dictionaries containing retrieved documents and metadata.
        """
        print(f"\nRetrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
        
        # 1. Generate query embedding
        # Fixed: Corrected spelling from 'quary' to 'query'
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # 2. Search in vector store 
        try:
            # Fixed: Corrected 'query_ebeddings' to 'query_embeddings' 
            # Fixed: Corrected 'n_result' to 'n_results' (ChromaDB standard)
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            # Fixed: ChromaDB returns lists within lists; correctly accessing keys
            if results['documents'] and len(results['documents'][0]) > 0:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]  # Fixed: 'metadatas' with 's'
                distances = results['distances'][0]  # Fixed: 'distances' with 's'
                ids = results['ids'][0]
                
                for i, (doc_id, doc_text, meta, dist) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB default is squared L2 distance)
                    # For Cosine Similarity, score would be 1 - distance
                    similarity_score = 1 - dist 
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': doc_text,
                            'metadata': meta,
                            'similarity_score': round(similarity_score, 4),
                            'distance': round(dist, 4),
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents after filtering.")
            else:
                print("No documents found in the vector store.")
            
            return retrieved_docs
                
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

# Initialize the retriever
# Assuming 'vector_store_manager' and 'embedding_manager' are already defined
rag_retriever = RAGRetriever(vector_store_manager, embedding_manager)

In [14]:
rag_retriever.retrieve("what is Attention Applications")


Retrieving documents for query: 'what is Attention Applications'
Top K: 5, Score Threshold: 0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents after filtering.


[{'id': 'doc_21e0a7df_77',
  'content': 'structure needed to properly distinguish the various atten-\ntion mechanisms. Additionally, certain signiﬁcant attention\ntechniques have not yet been properly discussed in previ-\nous surveys, while other presented attention mechanisms\nseem to be lacking either technical details or intuitive ex-\nplanations. Therefore, in this paper, we present important\nattention techniques by means of a single framework using\na uniform notation, a combination of both technical and in-\ntuitive explanations for each presented attention technique,\nand a comprehensive taxonomy of attention mechanisms.\nThe structure of this paper is as follows. Section 2 in-\ntroduces a general attention model that provides the reader\nwith a basic understanding of the properties of attention\nand how it can be applied. One of the main contributions\nof this paper is the taxonomy of attention techniques pre-\nsented in Section 3. In this section, attention mechanisms\nare ex

In [15]:
from typing import List, Dict, Any
import time 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key)
class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []
        
    def query(self, question: str, top_k: int = 5, min_score: float = 0.02, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # 1. Document Retrieval
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            # Extract content for context
            context = "\n\n".join([doc['content'] for doc in results])
            
            # Extract sources with metadata handling
            sources = []
            for doc in results:
                metadata = doc.get('metadata', {})
                sources.append({
                    'source': metadata.get('source_file', metadata.get('source', 'unknown')),
                    'page': metadata.get('page', 'unknown'),
                    'score': doc.get('similarity_score', 0),
                    'preview': doc['content'][:120] + '...'
                })
            
            # Construct Prompt
            prompt = f"Use the following context to answer the question concisely.\n\nContext:\n{context}\n\nQuestion: {question}"
            
            # Stream simulation
            if stream:
                print("Streaming answer (Preview):")
                for i in range(0, min(len(prompt), 500), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print("\n" + "-"*20)

            # 2. LLM Call for main answer
            response = self.llm.invoke(prompt)
            answer = response.content

        # 3. Add Citations
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        if citations:
            answer_with_citations = f"{answer}\n\nCitations:\n" + "\n".join(citations)
        else:
            answer_with_citations = answer

        # 4. Optional Summarization
        summary = None
        if summarize and results:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke(summary_prompt)
            summary = summary_resp.content
          
        # 5. Update History
        query_data = {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary
        }
        self.history.append(query_data)
        
        return {
            **query_data,
            'history': self.history
        }

# Example Usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is Evolution of Machine Learning?", top_k=3, min_score=0.1, stream=True, summarize=True)


Retrieving documents for query: 'What is Evolution of Machine Learning?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after filtering.
Streaming answer (Preview):
Use the following context to answer the question concisely.

Context:
programming. 
Machine learning is like farming or gardening. Seeds is the algorithms, nutrients is the data, the gardner is you and plants is the 
programs. 
 
Traditional Programming  vs Machine Learning 
Evolution of Machine Learning: 
Today, machi

ne learning is different from what it used to be in the past, due to the emergence of advanced computing 
technologies. Initially, it had gained momentum due to pattern recognition and the fact that computers did not have to be programed to
--------------------
